# Possession Parameter Sweep

**Purpose:** Sweep PossessionTracker's three parameters against the per-frame
possession ground truth, on both interpolation-filled and gated-only ball input, and
trace where the team possession-share error comes from.  
**Inputs:** `data/annotations/possession_gt_per_frame.csv`, the per-clip ball, player
and team caches in `data/processed/`, `config/default.yaml`.  
**Outputs:** `possession_sweep_results.csv`, `possession_sweep_gated_results.csv` and
`possession_share.csv` in `data/outputs/`, plus extracted phantom frames under
`data/outputs/clip_2_phantom_122_128/`.  
**Backs:** `results/possession/`.

Everything here is deterministic; there is no sampling and no seed to state. The first
cell pins the working directory to the repo root.

In [1]:
import os
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
os.chdir(REPO_ROOT)
print(os.getcwd())

/home/jovyan/nba-video-analytics


## 1. Ground truth and ball input

Loads the possession ground truth (unclear frames excluded from scoring; nobody
encoded as -1 to match the tracker's output), then each clip's cached ball detections
in both forms: gated (real detections only) and filled (gaps interpolated). The
real-anchor flags mark which filled frames carry a genuine detection.

In [2]:
import sys, itertools
import cv2, pathlib
from collections import Counter

import numpy as np
import pandas as pd
sys.path.insert(0, '.')

import yaml
from basketball.cache.cache_utils import load_cache
from basketball.possession.possession_io import possession_ball_input
from basketball.possession.possession_tracker import PossessionTracker
from basketball.utils.box_utils import bbox_center

CLIPS = ['clip_1', 'clip_2', 'clip_3']
GT_PATH = 'data/annotations/possession_gt_per_frame.csv'

with open('config/default.yaml') as f:
    base_config = yaml.safe_load(f)

gt = pd.read_csv(GT_PATH).drop_duplicates(subset=['clip', 'frame_idx'], keep='last')
n_unclear = (gt['holder'] == 'unclear').sum()
gt = gt[gt['holder'] != 'unclear'].copy()
# 'nobody' -> -1 so ground truth and tracker output share one encoding.
gt['true_holder'] = gt['holder'].apply(lambda h: -1 if h == 'nobody' else int(h))
print(f'{len(gt)} scoreable frames ({n_unclear} unclear excluded)')
print(gt.groupby('clip').size().to_string())

data = {}
for clip in CLIPS:
    raw_ball = load_cache(f'data/processed/{clip}/ball_detections.pkl')
    tracks = load_cache(f'data/processed/{clip}/player_detections.pkl')
    gated, filled = possession_ball_input(raw_ball)
    data[clip] = {'tracks': tracks, 'filled': filled,
                  'real': [f.get(1) is not None and f[1].confidence > 0.0 for f in filled]}
    print(f'{clip}: {len(filled)} frames, {sum(data[clip]["real"])} real-anchor '
          f'({sum(data[clip]["real"])/len(filled)*100:.0f}%)')

488 scoreable frames (46 unclear excluded)
clip
clip_1    102
clip_2    167
clip_3    219
clip_1: 117 frames, 80 real-anchor (68%)
clip_2: 174 frames, 90 real-anchor (52%)


clip_3: 243 frames, 150 real-anchor (62%)


## 2. Team labels and holder coverage

Establishes the shape of the cached team assignments, then measures how many
named-holder frames carry a team label for the holder's track. This bounds what the
possession-share attribution further down can use.

In [3]:
# --- structural probe: what does assign_teams actually emit? ---
ta = {}
for clip in CLIPS:
    path = f'data/processed/{clip}/team_assignment_fashionclip.pkl'
    print(f'{clip}: exists={os.path.exists(path)}')
    if os.path.exists(path):
        ta[clip] = load_cache(path)
        print(f'  type={type(ta[clip]).__name__} len={len(ta[clip])} '
              f'elem0_type={type(ta[clip][0]).__name__}')
        vals = Counter(v for frame in ta[clip] for v in frame.values())
        print(f'  team label values across all frames: {dict(vals)}')

# --- coverage: do the named-holder frames have team GT for the holder's track? ---
team_gt = pd.read_csv('data/annotations/team_assignment_gt_per_frame.csv')
team_gt = team_gt.drop_duplicates(subset=['clip', 'frame_idx', 'player_id'], keep='last')
print(f'\nteam GT rows after dedup: {len(team_gt)}')

named = gt[gt['true_holder'] != -1][['clip', 'frame_idx', 'true_holder']].copy()
named = named.rename(columns={'true_holder': 'player_id'})
print(f'named-holder frames (unclear already excluded): {len(named)}')
print(named.groupby('clip').size().to_string())

joined = named.merge(team_gt, on=['clip', 'frame_idx', 'player_id'], how='left')
joined['has_team_gt'] = joined['true_team'].notna()

print('\n=== holder-track team GT coverage, per clip ===')
print(joined.groupby('clip').agg(
    named_frames=('has_team_gt', 'size'),
    covered=('has_team_gt', 'sum'),
).assign(pct=lambda d: (100 * d.covered / d.named_frames).round(1)).to_string())

covered = joined[joined['has_team_gt']]
print(f'\ntotal covered: {len(covered)} of {len(named)}')
print('true_team values among covered:')
print(covered['true_team'].value_counts().to_string())

clip_1: exists=True
  type=list len=117 elem0_type=dict
  team label values across all frames: {2: 323, 1: 405}
clip_2: exists=True
  type=list len=174 elem0_type=dict
  team label values across all frames: {2: 737, 1: 886}
clip_3: exists=True
  type=list len=243 elem0_type=dict
  team label values across all frames: {1: 946, 2: 941}

team GT rows after dedup: 1238
named-holder frames (unclear already excluded): 360
clip
clip_1     89
clip_2    109
clip_3    162

=== holder-track team GT coverage, per clip ===
        named_frames  covered   pct
clip                               
clip_1            89       17  19.1
clip_2           109       22  20.2
clip_3           162       72  44.4

total covered: 111 of 360
true_team values among covered:
true_team
1          84
2          24
unclear     3


## 3. Scoring and the sweep

score() reports frame-level accuracy split by ball provenance, with recall on held
frames and specificity on nobody frames reported separately. The sweep covers every
combination of hold_threshold, max_ball_distance and bbox_overlap_min (240
configurations) on the interpolation-filled input.

In [4]:
def score(possession, clip, gt_sub, real_flags):
    """Frame-level accuracy against ground truth, split by ball-detection provenance."""
    m = gt_sub[gt_sub['clip'] == clip]
    rows = []
    for _, r in m.iterrows():
        i = int(r['frame_idx'])
        if i >= len(possession):
            continue
        rows.append({'pred': possession[i], 'true': int(r['true_holder']), 'real': real_flags[i]})
    d = pd.DataFrame(rows)
    if len(d) == 0:
        return {}
    d['correct'] = d['pred'] == d['true']

    held = d[d['true'] != -1]      # ground truth says someone holds it
    empty = d[d['true'] == -1]     # ground truth says nobody does
    return {
        'n': len(d),
        'acc': d['correct'].mean(),
        'recall_held': held['correct'].mean() if len(held) else np.nan,
        'spec_nobody': empty['correct'].mean() if len(empty) else np.nan,
        'false_pos': (empty['pred'] != -1).mean() if len(empty) else np.nan,
        'acc_real': d.loc[d['real'], 'correct'].mean() if d['real'].any() else np.nan,
        'acc_interp': d.loc[~d['real'], 'correct'].mean() if (~d['real']).any() else np.nan,
    }

In [5]:
HOLD = [1, 2, 3, 4, 5, 6, 8, 10]
DIST = [20, 30, 40, 50, 60, 80]
OVERLAP = [0.3, 0.5, 0.7, 0.8, 0.9]

rows = []
for hold, dist, ov in itertools.product(HOLD, DIST, OVERLAP):
    cfg = {**base_config, 'possession': {
        **base_config['possession'],
        'hold_threshold': hold, 'max_ball_distance': dist, 'bbox_overlap_min': ov}}
    tracker = PossessionTracker(cfg)
    for clip in CLIPS:
        d = data[clip]
        poss = tracker.assign_possession(d['tracks'], d['filled'], cache_path=None)
        s = score(poss, clip, gt, d['real'])
        if s:
            rows.append({'hold_threshold': hold, 'max_ball_distance': dist,
                         'bbox_overlap_min': ov, 'clip': clip, **s})

res = pd.DataFrame(rows)
res.to_csv('data/outputs/possession_sweep_results.csv', index=False)
print(f'{len(res)} (config, clip) results from {len(HOLD)*len(DIST)*len(OVERLAP)} configs')

720 (config, clip) results from 240 configs


## 4. Selection and the production configuration

Leave-one-clip-out selection (chosen on the two training clips, reported on the
held-out clip), the top configurations pooled across clips, the production
configuration's own scores, and each parameter's marginal effect.

In [6]:
pd.set_option('display.width', 220)
KEYS = ['hold_threshold', 'max_ball_distance', 'bbox_overlap_min']

folds = []
for held_out in CLIPS:
    train = res[res['clip'] != held_out].groupby(KEYS)['acc'].mean()
    best = train.idxmax()
    te = res[(res['clip'] == held_out) &
             (res['hold_threshold'] == best[0]) &
             (res['max_ball_distance'] == best[1]) &
             (res['bbox_overlap_min'] == best[2])].iloc[0]
    folds.append({'held_out': held_out, 'hold': best[0], 'dist': best[1], 'ov': best[2],
                  'train_acc': round(train.max(), 4), 'test_acc': round(te['acc'], 4),
                  'recall_held': round(te['recall_held'], 4),
                  'false_pos': round(te['false_pos'], 4),
                  'acc_real': round(te['acc_real'], 4),
                  'acc_interp': round(te['acc_interp'], 4)})

print('=== LEAVE-ONE-CLIP-OUT (selected on the two training clips) ===')
print(pd.DataFrame(folds).to_string(index=False))

pooled = res.groupby(KEYS)[['acc', 'recall_held', 'false_pos', 'acc_real', 'acc_interp']].mean()
print('\n=== TOP 10 CONFIGS (mean accuracy across all three clips) ===')
print(pooled.sort_values('acc', ascending=False).head(10).round(4).to_string())

print('\n=== Current production config ===')
p = base_config['possession']
cur = res[(res['hold_threshold'] == p['hold_threshold']) &
          (res['max_ball_distance'] == p['max_ball_distance']) &
          (res['bbox_overlap_min'] == p['bbox_overlap_min'])]
print(cur[['clip', 'n', 'acc', 'recall_held', 'false_pos', 'acc_real', 'acc_interp']].round(4).to_string(index=False))

print('\n=== Marginal effect of each parameter (mean acc, others averaged) ===')
for k in KEYS:
    print(f'\n{k}:')
    print(res.groupby(k)['acc'].mean().round(4).to_string())

=== LEAVE-ONE-CLIP-OUT (selected on the two training clips) ===
held_out  hold  dist  ov  train_acc  test_acc  recall_held  false_pos  acc_real  acc_interp
  clip_1     2    30 0.3     0.6430    0.8235       0.7978     0.0000    0.8592      0.7419
  clip_2     2    60 0.7     0.8062    0.5329       0.6422     0.6724    0.6118      0.4512
  clip_3     8    60 0.3     0.7700    0.5571       0.6358     0.6667    0.5000      0.6667

=== TOP 10 CONFIGS (mean accuracy across all three clips) ===
                                                      acc  recall_held  false_pos  acc_real  acc_interp
hold_threshold max_ball_distance bbox_overlap_min                                                      
3              50                0.7               0.7258       0.7668     0.3794    0.7658      0.6826
                                 0.3               0.7237       0.7647     0.3794    0.7644      0.6826
                                 0.5               0.7237       0.7647     0.3794    0.76

## 5. False positives against adjacency

For frames ground truth says nobody held, how far is each false positive from the
nearest genuinely held frame? Adjacent false positives are boundary disagreements;
distant ones are phantom possessions.

In [7]:
p = base_config['possession']
tracker = PossessionTracker(base_config)

for clip in CLIPS:
    d = data[clip]
    poss = tracker.assign_possession(d['tracks'], d['filled'], cache_path=None)
    g = gt[gt['clip'] == clip].set_index('frame_idx')['true_holder'].to_dict()

    # Distance (in frames) from each "nobody" frame to the nearest frame that
    # ground truth says was genuinely held. 1 = adjacent to a real possession.
    held_frames = sorted(f for f, v in g.items() if v != -1)
    fp_dist, tn_dist = [], []
    for f, true in g.items():
        if true != -1 or f >= len(poss):
            continue
        nearest = min((abs(f - h) for h in held_frames), default=999)
        (fp_dist if poss[f] != -1 else tn_dist).append(nearest)

    def summarise(xs):
        if not xs:
            return 'none'
        a = np.array(xs)
        return (f'n={len(a)} median={np.median(a):.0f} '
                f'within3={100*(a <= 3).mean():.0f}% within8={100*(a <= 8).mean():.0f}%')

    print(f'{clip}')
    print(f'  false positives : {summarise(fp_dist)}')
    print(f'  correct nobody  : {summarise(tn_dist)}')

clip_1
  false positives : n=1 median=5 within3=0% within8=100%
  correct nobody  : n=12 median=8 within3=0% within8=50%
clip_2
  false positives : n=24 median=3 within3=58% within8=100%
  correct nobody  : n=34 median=5 within3=24% within8=82%
clip_3
  false positives : n=30 median=12 within3=20% within8=37%
  correct nobody  : n=27 median=4 within3=37% within8=70%


## 6. The raw holder ceiling

find_holder() alone, with no streak logic and no backfill: the ceiling any state
machine built on it can reach, compared against the full tracker at the production
configuration.

In [8]:
print('Raw find_holder(), no streak logic, no backfill — the ceiling any')
print('state machine on top of it can reach.\n')

rows = []
for dist in [30, 50, 80]:
    cfg = {**base_config, 'possession': {**p, 'max_ball_distance': dist}}
    t = PossessionTracker(cfg)
    for clip in CLIPS:
        d = data[clip]
        g = gt[gt['clip'] == clip].set_index('frame_idx')['true_holder'].to_dict()
        n = correct = held_ok = held_n = fp = empty_n = 0
        for f, true in g.items():
            if f >= len(d['filled']):
                continue
            ball = d['filled'][f].get(1)
            pred = -1 if ball is None or not ball.bbox else t.find_holder(
                bbox_center(*ball.bbox), d['tracks'][f], ball.bbox)
            n += 1
            correct += (pred == true)
            if true != -1:
                held_n += 1
                held_ok += (pred == true)
            else:
                empty_n += 1
                fp += (pred != -1)
        rows.append({'max_ball_distance': dist, 'clip': clip, 'n': n,
                     'raw_acc': correct / n,
                     'raw_recall_held': held_ok / held_n if held_n else np.nan,
                     'raw_false_pos': fp / empty_n if empty_n else np.nan})

raw = pd.DataFrame(rows)
print(raw.round(4).to_string(index=False))

print('\n=== Raw find_holder vs the full tracker (production config) ===')
prod = res[(res['hold_threshold'] == p['hold_threshold']) &
           (res['max_ball_distance'] == p['max_ball_distance']) &
           (res['bbox_overlap_min'] == p['bbox_overlap_min'])][['clip', 'acc', 'false_pos']]
cmp = raw[raw['max_ball_distance'] == p['max_ball_distance']][['clip', 'raw_acc', 'raw_false_pos']]
print(cmp.merge(prod, on='clip').round(4).to_string(index=False))

Raw find_holder(), no streak logic, no backfill — the ceiling any
state machine on top of it can reach.



 max_ball_distance   clip   n  raw_acc  raw_recall_held  raw_false_pos
                30 clip_1 102   0.8529           0.8315         0.0000
                30 clip_2 167   0.6168           0.6697         0.4828
                30 clip_3 219   0.6621           0.7654         0.6316
                50 clip_1 102   0.9608           0.9663         0.0769
                50 clip_2 167   0.5389           0.7064         0.7759
                50 clip_3 219   0.6575           0.8210         0.8070
                80 clip_1 102   0.9412           0.9775         0.3077
                80 clip_2 167   0.4910           0.7248         0.9483
                80 clip_3 219   0.6804           0.8889         0.9123

=== Raw find_holder vs the full tracker (production config) ===
  clip  raw_acc  raw_false_pos    acc  false_pos
clip_1   0.9608         0.0769 0.9314     0.0769
clip_2   0.5389         0.7759 0.5928     0.4138
clip_3   0.6575         0.8070 0.6256     0.5263


## 7. Gated-only arm

The same sweep on gated input only, so interpolation's contribution is measured rather
than assumed. The second cell repeats the leave-one-clip-out selection and compares
filled against gated at the production configuration.

In [9]:
data_gated = {}
for clip in CLIPS:
    d = data[clip]
    gated_dict = {i: ({1: f[1]} if f.get(1) is not None else {})
                  for i, f in enumerate(d['filled'])}
    # Recover the actual gated (pre-fill) detections directly, not by
    # guessing from 'filled': filled's confidence==0 marks interpolated
    # frames, but gated's OWN dict is the real source of truth for which
    # frames had a genuine detection at all.
    raw_ball = load_cache(f'data/processed/{clip}/ball_detections.pkl')
    gated, _ = possession_ball_input(raw_ball)
    data_gated[clip] = {'tracks': d['tracks'], 'gated': gated}
    n_present = sum(1 for f in gated if f.get(1) is not None)
    print(f'{clip}: {n_present}/{len(gated)} frames with a real ball detection '
          f'({n_present/len(gated)*100:.0f}%)')

rows_g = []
for hold, dist, ov in itertools.product(HOLD, DIST, OVERLAP):
    cfg = {**base_config, 'possession': {
        **base_config['possession'],
        'hold_threshold': hold, 'max_ball_distance': dist, 'bbox_overlap_min': ov}}
    tracker = PossessionTracker(cfg)
    for clip in CLIPS:
        d = data_gated[clip]
        poss = tracker.assign_possession(d['tracks'], d['gated'], cache_path=None)
        real_flags = [True] * len(poss)  # every scored frame here IS a real anchor by construction
        s = score(poss, clip, gt, real_flags)
        if s:
            rows_g.append({'hold_threshold': hold, 'max_ball_distance': dist,
                           'bbox_overlap_min': ov, 'clip': clip, **s})

res_g = pd.DataFrame(rows_g)
res_g.to_csv('data/outputs/possession_sweep_gated_results.csv', index=False)
print(f'\n{len(res_g)} (config, clip) results on gated-only input')

clip_1: 80/117 frames with a real ball detection (68%)
clip_2: 90/174 frames with a real ball detection (52%)
clip_3: 150/243 frames with a real ball detection (62%)



720 (config, clip) results on gated-only input


In [10]:
folds_g = []
for held_out in CLIPS:
    train = res_g[res_g['clip'] != held_out].groupby(KEYS)['acc'].mean()
    best = train.idxmax()
    te = res_g[(res_g['clip'] == held_out) &
               (res_g['hold_threshold'] == best[0]) &
               (res_g['max_ball_distance'] == best[1]) &
               (res_g['bbox_overlap_min'] == best[2])].iloc[0]
    folds_g.append({'held_out': held_out, 'hold': best[0], 'dist': best[1], 'ov': best[2],
                    'train_acc': round(train.max(), 4), 'test_acc': round(te['acc'], 4),
                    'recall_held': round(te['recall_held'], 4),
                    'false_pos': round(te['false_pos'], 4),
                    'n': int(te['n'])})

print('=== GATED-ONLY: LEAVE-ONE-CLIP-OUT ===')
print(pd.DataFrame(folds_g).to_string(index=False))

print('\n=== BEFORE (filled) vs AFTER (gated) — production hold/ov, dist=50 ===')
p = base_config['possession']
before = res[(res['hold_threshold'] == p['hold_threshold']) &
             (res['max_ball_distance'] == p['max_ball_distance']) &
             (res['bbox_overlap_min'] == p['bbox_overlap_min'])][['clip', 'n', 'acc', 'false_pos', 'recall_held']]
after = res_g[(res_g['hold_threshold'] == p['hold_threshold']) &
              (res_g['max_ball_distance'] == p['max_ball_distance']) &
              (res_g['bbox_overlap_min'] == p['bbox_overlap_min'])][['clip', 'n', 'acc', 'false_pos', 'recall_held']]
cmp = before.merge(after, on='clip', suffixes=('_filled', '_gated'))
print(cmp.round(4).to_string(index=False))

print('\n=== Marginal effect of max_ball_distance, gated-only ===')
print(res_g.groupby('max_ball_distance')['acc'].mean().round(4).to_string())

=== GATED-ONLY: LEAVE-ONE-CLIP-OUT ===
held_out  hold  dist  ov  train_acc  test_acc  recall_held  false_pos   n
  clip_1    10    60 0.7     0.5978    0.6373       0.5843     0.0000 102
  clip_2     8    60 0.7     0.6821    0.5928       0.4128     0.0690 167
  clip_3     5    60 0.3     0.6986    0.4795       0.5000     0.5789 219

=== BEFORE (filled) vs AFTER (gated) — production hold/ov, dist=50 ===
  clip  n_filled  acc_filled  false_pos_filled  recall_held_filled  n_gated  acc_gated  false_pos_gated  recall_held_gated
clip_1       102      0.9314            0.0769              0.9326      102     0.7647           0.0000             0.7303
clip_2       167      0.5928            0.4138              0.5963      167     0.6168           0.0690             0.4495
clip_3       219      0.6256            0.5263              0.6790      219     0.5297           0.4561             0.5247

=== Marginal effect of max_ball_distance, gated-only ===
max_ball_distance
20    0.5229
30    0.5854

## 8. Held fraction by gap length

How often ground truth says the ball was genuinely held, bucketed by the length of the
detection gap the frame sits in. Long gaps are where interpolated anchors are least
trustworthy.

In [11]:
def gap_lengths(filled_frames):
    """For each frame, the length of the detection gap it falls inside
    (0 if it's a real detection, else the number of consecutive
    interpolated frames in that run — same value for every frame in the run)."""
    n = len(filled_frames)
    is_real = [f.get(1) is not None and f[1].confidence > 0.0 for f in filled_frames]
    lengths = [0] * n
    i = 0
    while i < n:
        if is_real[i]:
            i += 1
            continue
        j = i
        while j < n and not is_real[j]:
            j += 1
        for k in range(i, j):
            lengths[k] = j - i
        i = j
    return lengths

rows_gap = []
for clip in CLIPS:
    d = data[clip]
    lens = gap_lengths(d['filled'])
    g = gt[gt['clip'] == clip].set_index('frame_idx')['true_holder'].to_dict()
    for f, true in g.items():
        if f >= len(lens):
            continue
        rows_gap.append({'clip': clip, 'frame_idx': f, 'gap_len': lens[f],
                         'is_real': lens[f] == 0, 'true_held': true != -1})

gap_df = pd.DataFrame(rows_gap)

print('=== Fraction of ground-truth-HELD frames, by gap length bucket ===')
bins = [-1, 0, 2, 5, 10, 20, 999]
labels = ['real(0)', '1-2', '3-5', '6-10', '11-20', '20+']
gap_df['bucket'] = pd.cut(gap_df['gap_len'], bins=bins, labels=labels)

summary = gap_df.groupby('bucket', observed=True).agg(
    n=('true_held', 'size'),
    pct_held=('true_held', 'mean')
)
print(summary.round(3).to_string())

print('\n=== Same, split by clip ===')
for clip in CLIPS:
    sub = gap_df[gap_df['clip'] == clip]
    s = sub.groupby('bucket', observed=True).agg(n=('true_held', 'size'), pct_held=('true_held', 'mean'))
    print(f'\n{clip}:')
    print(s.round(3).to_string())

print('\n=== Gap length distribution itself (how common is each bucket) ===')
print(gap_df['bucket'].value_counts().sort_index().to_string())

=== Fraction of ground-truth-HELD frames, by gap length bucket ===
           n  pct_held
bucket                
real(0)  300     0.770
1-2       47     0.766
3-5       52     0.731
6-10      47     0.553
11-20     42     0.690

=== Same, split by clip ===

clip_1:
          n  pct_held
bucket               
real(0)  71     0.887
1-2       6     1.000
3-5      13     1.000
6-10     12     0.583

clip_2:
          n  pct_held
bucket               
real(0)  85     0.776
1-2      19     0.737
3-5      16     0.375
6-10     14     0.214
11-20    33     0.606

clip_3:
           n  pct_held
bucket                
real(0)  144     0.708
1-2       22     0.727
3-5       23     0.826
6-10      21     0.762
11-20      9     1.000

=== Gap length distribution itself (how common is each bucket) ===
bucket
real(0)    300
1-2         47
3-5         52
6-10        47
11-20       42
20+          0


## 9. Possession share

The pipeline's headline output: team share of attributed frames, computed on predicted
possession over the full clips, predicted possession restricted to scoreable frames,
and oracle possession on the same frames. A holder without a team label of 1 or 2 is
unattributed, never silently a team.

In [12]:
def compute_share(holders, team_assign, frame_indices):
    """Possession share over attributed frames, with attribution reported separately.

    holders: dict frame_idx -> holder track id, -1 for nobody.
    team_assign: list of dict track_id -> team label, one entry per frame.
    A frame is attributed only when a named holder carries a label of 1 or 2 —
    a holder absent from the frame's assignment, or carrying any other label
    (including an abstention 0), is unattributed, never silently team 2.
    """
    t1 = t2 = no_holder = holder_unlabelled = 0
    for f in frame_indices:
        if f >= len(team_assign):
            continue
        h = holders.get(f, -1)
        if h == -1:
            no_holder += 1
            continue
        label = team_assign[f].get(h)
        if label == 1:
            t1 += 1
        elif label == 2:
            t2 += 1
        else:
            holder_unlabelled += 1

    attributed = t1 + t2
    total = attributed + no_holder + holder_unlabelled
    return {
        'n_frames': total,
        'n_team_1': t1,
        'n_team_2': t2,
        'n_no_holder': no_holder,
        'n_holder_unlabelled': holder_unlabelled,
        'attribution_rate': attributed / total if total else float('nan'),
        'team_1_share': t1 / attributed if attributed else float('nan'),
        'team_2_share': t2 / attributed if attributed else float('nan'),
    }

tracker = PossessionTracker(base_config)
rows = []
pred_holders_by_clip, team_by_clip, scoreable_by_clip = {}, {}, {}

for clip in CLIPS:
    d = data[clip]
    poss = tracker.assign_possession(d['tracks'], d['filled'], cache_path=None)
    team = load_cache(f'data/processed/{clip}/team_assignment_fashionclip.pkl')

    g = gt[gt['clip'] == clip].set_index('frame_idx')['true_holder'].to_dict()
    scoreable = sorted(f for f in g if f < len(poss))

    pred_holders = {f: poss[f] for f in range(len(poss))}
    true_holders = {f: g[f] for f in scoreable}

    pred_holders_by_clip[clip] = pred_holders
    team_by_clip[clip] = team
    scoreable_by_clip[clip] = scoreable

    rows.append({'clip': clip, 'basis': 'predicted_full',
                 **compute_share(pred_holders, team, range(len(poss)))})
    rows.append({'clip': clip, 'basis': 'predicted_restricted',
                 **compute_share(pred_holders, team, scoreable)})
    rows.append({'clip': clip, 'basis': 'oracle_possession',
                 **compute_share(true_holders, team, scoreable)})

shares = pd.DataFrame(rows)
pd.set_option('display.width', 220)
print(shares.round(4).to_string(index=False))
shares.to_csv('data/outputs/possession_share.csv', index=False)

  clip                basis  n_frames  n_team_1  n_team_2  n_no_holder  n_holder_unlabelled  attribution_rate  team_1_share  team_2_share
clip_1       predicted_full       117         8        83           26                    0            0.7778        0.0879        0.9121
clip_1 predicted_restricted       102         4        82           16                    0            0.8431        0.0465        0.9535
clip_1    oracle_possession       102         5        84           13                    0            0.8725        0.0562        0.9438
clip_2       predicted_full       174        78        33           63                    0            0.6379        0.7027        0.2973
clip_2 predicted_restricted       167        77        33           57                    0            0.6587        0.7000        0.3000
clip_2    oracle_possession       167       108         1           58                    0            0.6527        0.9908        0.0092
clip_3       predicted_full       

## 10. Share error and phantoms

Restricting predicted and oracle possession to the same frames and the same team
labels isolates the share error attributable to possession detection alone.

In [13]:
print('=== Possession share error attributable to possession detection ===')
print('(predicted_restricted vs oracle_possession, same frames, same team labels)\n')

out = []
for clip in CLIPS:
    p = shares[(shares['clip'] == clip) & (shares['basis'] == 'predicted_restricted')].iloc[0]
    o = shares[(shares['clip'] == clip) & (shares['basis'] == 'oracle_possession')].iloc[0]
    f = shares[(shares['clip'] == clip) & (shares['basis'] == 'predicted_full')].iloc[0]
    out.append({
        'clip': clip,
        'pred_t1_share': round(p.team_1_share, 4),
        'oracle_t1_share': round(o.team_1_share, 4),
        'abs_error_pp': round(abs(p.team_1_share - o.team_1_share) * 100, 2),
        'pred_attr_rate': round(p.attribution_rate, 4),
        'oracle_attr_rate': round(o.attribution_rate, 4),
        'full_clip_t1_share': round(f.team_1_share, 4),
        'full_clip_attr_rate': round(f.attribution_rate, 4),
    })
print(pd.DataFrame(out).to_string(index=False))


=== Possession share error attributable to possession detection ===
(predicted_restricted vs oracle_possession, same frames, same team labels)

  clip  pred_t1_share  oracle_t1_share  abs_error_pp  pred_attr_rate  oracle_attr_rate  full_clip_t1_share  full_clip_attr_rate
clip_1         0.0465           0.0562          0.97          0.8431            0.8725              0.0879               0.7778
clip_2         0.7000           0.9908         29.08          0.6587            0.6527              0.7027               0.6379
clip_3         0.7027           0.7284          2.57          0.6758            0.7397              0.6845               0.6914


Do phantom holders carry the same team as the nearest genuine holder? Where they do, a
phantom moves the share only slightly; where they do not, it shifts the share toward
the wrong team.

In [14]:
print('=== Team of phantom holders vs team of the nearest genuine holder ===\n')

for clip in CLIPS:
    g = gt[gt['clip'] == clip].set_index('frame_idx')['true_holder'].to_dict()
    team = team_by_clip[clip]
    pred = pred_holders_by_clip[clip]
    held_frames = sorted(f for f, v in g.items() if v != -1)

    same = diff = undetermined = 0
    for f in scoreable_by_clip[clip]:
        if g[f] != -1 or pred[f] == -1:
            continue                      # not a phantom
        phantom_team = team[f].get(pred[f])
        nearest = min(held_frames, key=lambda h: abs(f - h), default=None)
        true_team = team[nearest].get(g[nearest]) if nearest is not None else None
        if phantom_team in (1, 2) and true_team in (1, 2):
            same += (phantom_team == true_team)
            diff += (phantom_team != true_team)
        else:
            undetermined += 1

    n = same + diff
    pct = f'{100 * same / n:.0f}%' if n else 'n/a'
    print(f'{clip}: phantoms={n + undetermined}  comparable={n}  '
          f'same team as nearest true holder={pct}  undetermined={undetermined}')

=== Team of phantom holders vs team of the nearest genuine holder ===

clip_1: phantoms=1  comparable=1  same team as nearest true holder=100%  undetermined=0
clip_2: phantoms=24  comparable=24  same team as nearest true holder=50%  undetermined=0
clip_3: phantoms=30  comparable=30  same team as nearest true holder=70%  undetermined=0


Counts each disagreement type between predicted and oracle possession on the scoreable
frames: false positives, misses, wrong players, and the two correct kinds. The balance
explains why the attribution rates differ.

In [15]:
print('=== Why predicted and oracle attribution rates differ ===')
print('false_positive: GT nobody, pipeline names someone')
print('miss:           GT names someone, pipeline says nobody\n')

rows = []
for clip in CLIPS:
    g = gt[gt['clip'] == clip].set_index('frame_idx')['true_holder'].to_dict()
    pred = pred_holders_by_clip[clip]
    fp = miss = correct_held = correct_nobody = wrong_player = 0
    for f in scoreable_by_clip[clip]:
        t, p = g[f], pred[f]
        if t == -1 and p != -1:
            fp += 1
        elif t != -1 and p == -1:
            miss += 1
        elif t != -1 and p == t:
            correct_held += 1
        elif t != -1:
            wrong_player += 1
        else:
            correct_nobody += 1
    rows.append({'clip': clip, 'n': len(scoreable_by_clip[clip]),
                 'false_positive': fp, 'miss': miss, 'wrong_player': wrong_player,
                 'correct_held': correct_held, 'correct_nobody': correct_nobody,
                 'net_attr_change': fp - miss - wrong_player * 0})
print(pd.DataFrame(rows).to_string(index=False))

=== Why predicted and oracle attribution rates differ ===
false_positive: GT nobody, pipeline names someone
miss:           GT names someone, pipeline says nobody

  clip   n  false_positive  miss  wrong_player  correct_held  correct_nobody  net_attr_change
clip_1 102               1     4             2            83              12               -3
clip_2 167              24    23            21            65              34                1
clip_3 219              30    44             8           110              27              -14


## 11. The clip_2 phantom run

clip_2's phantoms carry a 29-point share error, so each one is listed with its team,
its ball provenance and its distance to the nearest genuinely held frame.

In [16]:
CLIP = 'clip_2'
g = gt[gt['clip'] == CLIP].set_index('frame_idx')['true_holder'].to_dict()
team = team_by_clip[CLIP]
pred = pred_holders_by_clip[CLIP]
real = data[CLIP]['real']
held_frames = sorted(f for f, v in g.items() if v != -1)

rows = []
for f in scoreable_by_clip[CLIP]:
    if g[f] != -1 or pred[f] == -1:
        continue
    h = pred[f]
    nearest = min(held_frames, key=lambda x: abs(f - x), default=None)
    rows.append({
        'frame': f,
        'pred_holder': h,
        'pred_team': team[f].get(h),
        'ball_real': real[f],
        'nearest_true_frame': nearest,
        'frames_away': abs(f - nearest) if nearest is not None else None,
        'true_holder_there': g[nearest] if nearest is not None else None,
        'its_team': team[nearest].get(g[nearest]) if nearest is not None else None,
    })

ph = pd.DataFrame(rows).sort_values('frame')
print(f'All {len(ph)} phantoms on {CLIP}:\n')
print(ph.to_string(index=False))
print(f'\nThe {(ph.pred_team == 2).sum()} attributed to team 2 '
      f'(against 1 genuine team-2 frame) carry the 29pp error:')
print(ph[ph.pred_team == 2]['frame'].tolist())

All 24 phantoms on clip_2:

 frame  pred_holder  pred_team  ball_real  nearest_true_frame  frames_away  true_holder_there  its_team
    43            4          2      False                  44            1                  2         1
    53            4          2      False                  52            1                  2         1
    54            4          2      False                  52            2                  2         1
    55            4          2      False                  52            3                  2         1
    56            4          2      False                  52            4                  2         1
    91            7          2      False                  90            1                  7         2
    92            7          2      False                  90            2                  7         2
    93            7          2      False                  90            3                  7         2
   122            1          2      

Extracts the frames around the 122 to 128 phantom run, with context either side, from
the annotated clip_2 render: the visual evidence of what the tracker latched onto.

In [17]:
VIDEO = 'data/outputs/clip_2_annotated.avi'
OUT = pathlib.Path('data/outputs/clip_2_phantom_122_128')
OUT.mkdir(parents=True, exist_ok=True)
WANT = list(range(118, 133))   # the run, plus context either side

cap = cv2.VideoCapture(VIDEO)
if not cap.isOpened():
    raise FileNotFoundError(f'Cannot open {VIDEO}')

g = gt[gt['clip'] == 'clip_2'].set_index('frame_idx')['true_holder'].to_dict()
pred = pred_holders_by_clip['clip_2']
team = team_by_clip['clip_2']
real = data['clip_2']['real']

saved, idx = 0, 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    if idx in WANT:
        cv2.imwrite(str(OUT / f'frame_{idx:03d}.png'), frame)
        saved += 1
    idx += 1
cap.release()
print(f'{saved} frames written to {OUT}  (video had {idx} frames)\n')

print('frame  gt_holder  pred_holder  pred_team  ball_real')
for f in WANT:
    t = g.get(f, 'not-scoreable')
    p = pred.get(f)
    print(f'{f:5d}  {str(t):>9}  {str(p):>11}  '
          f'{str(team[f].get(p) if p not in (None, -1) else "-"):>9}  {real[f]}')

15 frames written to data/outputs/clip_2_phantom_122_128  (video had 174 frames)

frame  gt_holder  pred_holder  pred_team  ball_real
  118         10            1          2  False
  119         10            1          2  False
  120         10            1          2  False
  121         10            1          2  False
  122         -1            1          2  False
  123         -1            1          2  False
  124         -1            1          2  False
  125         -1            1          2  False
  126         -1            1          2  True
  127         -1            1          2  False
  128         -1            1          2  False
  129         -1            7          1  True
  130         -1            7          1  True
  131         -1            7          1  True
  132         -1            7          1  True


## 12. Outcome

Two sweep CSVs (`possession_sweep_results.csv`, `possession_sweep_gated_results.csv`)
and `possession_share.csv` are written to `data/outputs/`, with phantom frames under
`data/outputs/clip_2_phantom_122_128/`. The copies shipped with the repository are in
`results/possession/`.